# 夜光遥感数据预处理系统

本 Notebook 调用 `functions/preprocessing.py` 模块完成 VNP46A1/A2 数据预处理。

**功能说明：**
- Stage1: 从 HDF5 文件提取夜光亮度、观测角度、质量标志图层（支持提取时直接裁剪）
- Stage2: 生成标准化的时间序列文本数据（自动镶嵌跨瓦片研究区）
- 可选: 使用 Shapefile 裁剪研究区、多瓦片自动镶嵌

In [ ]:
# 导入预处理模块（使用 reload 以便在模块编辑后生效）
import sys, os
from importlib import import_module, reload

# 添加项目路径
cwd = os.getcwd()
project_root = cwd if os.path.basename(cwd) == 'ntl_prophet' else os.path.join(cwd, 'ntl_prophet')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 载入并重载模块以读取最新更改
try:
    import functions.preprocessing as preprocessing_module
    reload(preprocessing_module)
except Exception:
    preprocessing_module = import_module('functions.preprocessing')
    reload(preprocessing_module)

# 从模块导入需要的函数（导入后可直接调用）
from functions.preprocessing import (
    stage1_extract_and_pair,
    stage2_generate_time_series,
    clip_rasters_by_shapefile,
    mosaic_tiles_by_date,
    complete_ntl_preprocessing_pipeline
)

print('✅ 预处理模块加载成功')


## 配置参数

根据实际数据路径修改以下配置项。

In [ ]:

# ============== 配置区域 ==============
# 输入路径
INPUT_VNP46A2_FOLDER = r'.\datasets\VNP46A2'   # VNP46A2 文件夹
INPUT_VNP46A1_FOLDER = r'.\datasets\VNP46A1'   # VNP46A1 文件夹

# 输出路径
OUTPUT_BASE_FOLDER = r'.\processed'
FINAL_OUTPUT_FILE = r'.\processed\ntl_timeseries.txt'

# Zenith 填充策略: 'zero' 或 'mean'
ZENITH_FILL_MODE = 'zero'

# ---- 时间范围过滤 ----
# 格式 'YYYY-MM-DD'，设为 None 时不过滤
DATE_START = '2023-01-01'          # 例如 '2020-01-01'
DATE_END   = '2025-01-01'          # 例如 '2023-12-31'

# ---- 瓦片编号过滤 ----
# 设为 None 时处理所有瓦片；否则填写瓦片列表
# 支持字符串格式（如 'h27v05'）或元组格式（如 (27, 5)）
TILE_FILTER = 'h21v05'         # 例如 ['h27v05', 'h28v05']

# ---- 裁剪相关配置 ----
USE_SHAPE_CLIP = True  # 是否使用 Shapefile 裁剪
# 是否用 shapefile 的外接矩形 (bounding box) 进行裁剪，True 则使用 bbox 裁剪（速度更快，但边界较粗）
USE_SHAPE_CLIP_BY_BBOX = False
SHAPEFILE_PATH = r'.\datasets\shp\AOI.shp'
CLIP_PROCESSES = 4     # 并行进程数
CLIP_OVERWRITE = False
CLIP_WARP_OPTIONS = {'dstNodata': -9999}
CLIP_PARALLEL_MODE = 'auto'  # 'auto'|'process'|'thread'|'none'

# 是否清理中间文件
CLEANUP_INTERMEDIATE = True

print('🔧 当前配置:')
print(f'   VNP46A2: {INPUT_VNP46A2_FOLDER}')
print(f'   VNP46A1: {INPUT_VNP46A1_FOLDER}')
print(f'   输出目录: {OUTPUT_BASE_FOLDER}')
print(f'   输出文件: {FINAL_OUTPUT_FILE}')
print(f'   Zenith填充: {ZENITH_FILL_MODE}')
print(f"   时间范围: {DATE_START or '不限'} ~ {DATE_END or '不限'}")
print(f"   瓦片过滤: {TILE_FILTER or '全部'}")
print(f'   使用裁剪: {USE_SHAPE_CLIP}')
if USE_SHAPE_CLIP:
    print(f'   裁剪矢量: {SHAPEFILE_PATH}')


## 执行数据预处理

使用 `complete_ntl_preprocessing_pipeline()` 一键完成全流程。

In [ ]:
# 检查输入路径
vnp46a2_exists = os.path.exists(INPUT_VNP46A2_FOLDER)
vnp46a1_exists = os.path.exists(INPUT_VNP46A1_FOLDER)

print('📁 文件夹检查:')
print(f"   VNP46A2: {'✅' if vnp46a2_exists else '❌'} {INPUT_VNP46A2_FOLDER}")
print(f"   VNP46A1: {'✅' if vnp46a1_exists else '❌'} {INPUT_VNP46A1_FOLDER}")

if not vnp46a2_exists:
    raise FileNotFoundError('VNP46A2 文件夹不存在，请检查路径配置')


In [ ]:
# 一键运行完整预处理流程
result = complete_ntl_preprocessing_pipeline(
    input_vnp46a2_folder=INPUT_VNP46A2_FOLDER,
    input_vnp46a1_folder=INPUT_VNP46A1_FOLDER,
    output_base_folder=OUTPUT_BASE_FOLDER,
    final_output_file=FINAL_OUTPUT_FILE,
    cleanup_intermediate=CLEANUP_INTERMEDIATE,
    zenith_fill_mode=ZENITH_FILL_MODE,
    use_shape_clip=USE_SHAPE_CLIP,
    shapefile_path=SHAPEFILE_PATH if USE_SHAPE_CLIP else None,
    clip_processes=CLIP_PROCESSES,
    clip_overwrite=CLIP_OVERWRITE,
    clip_warp_options=CLIP_WARP_OPTIONS,
    clip_parallel_mode=CLIP_PARALLEL_MODE,
    use_shapefile_bbox=USE_SHAPE_CLIP_BY_BBOX,
    date_start=DATE_START,
    date_end=DATE_END,
    tile_filter=TILE_FILTER
)

print('
✅ 预处理完成!')
print(f'   输出文件: {FINAL_OUTPUT_FILE}')
